In [2]:
# Otonom sürüşlü taksi- 5x5 alan ve 4 durak

In [5]:
import gymnasium as gym

# render_mode="ansi" görüntüdeki metin çıktısını sağlar
env = gym.make("Taxi-v3", render_mode="ansi")

state, info = env.reset()
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [8]:
import numpy as np

# Durum ve aksiyon sayılarını alarak tabloyu sıfırlarla doldur
q_table = np.zeros([env.observation_space.n, env.action_space.n])

print(f"Tablo Yapısı: {q_table.shape}") # (500, 6) olmalı

Tablo Yapısı: (500, 6)


In [11]:
import random

# Hiperparametreler
alpha = 0.1    # Öğrenme hızı
gamma = 0.6    # İndirim faktörü
epsilon = 0.1  # Keşif (Exploration) oranı

for i in range(1, 100001):
    state, info = env.reset()
    
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        # Epsilon-greedy stratejisi
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Keşfet
        else:
            action = np.argmax(q_table[state]) # Öğrenileni yap
            
        # DİKKAT: Gymnasium 5 değer döndürür
        next_state, reward, terminated, truncated, info = env.step(action) 
        
        # Q-Learning Formülü
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
        
        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state, action] = new_value
        
        state = next_state
        
    if i % 10000 == 0:
        print(f"Bölüm: {i}")

print("Eğitim tamamlandı.\n")

Bölüm: 10000
Bölüm: 20000
Bölüm: 30000
Bölüm: 40000
Bölüm: 50000
Bölüm: 60000
Bölüm: 70000
Bölüm: 80000
Bölüm: 90000
Bölüm: 100000
Eğitim tamamlandı.



In [13]:
# Sonuçları izleme 1.durum

In [16]:
from IPython.display import clear_output
from time import sleep

total_epochs, total_penalties = 0, 0
episodes = 5

for ep in range(episodes):
    state, info = env.reset()
    epochs, penalties, reward = 0, 0, 0
    terminated, truncated = False, False
    
    while not (terminated or truncated):
        action = np.argmax(q_table[state])
        state, reward, terminated, truncated, info = env.step(action)

        if reward == -10:
            penalties += 1

        # EKRAN ÇIKTISI (İstediğin format)
        clear_output(wait=True)
        print(env.render())
        print(f"Timestep: {epochs}")
        print(f"State: {state}")
        print(f"Action: {action}")
        print(f"Reward: {reward}")
        
        epochs += 1
        sleep(0.1) # İzlenebilmesi için yavaşlatıyoruz

    total_epochs += epochs
    total_penalties += penalties

print(f"\nSonuçlar {episodes} bölüm üzerinden:")
print(f"Ortalama adım: {total_epochs / episodes}")
print(f"Ortalama ceza: {total_penalties / episodes}")

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

Timestep: 14
State: 85
Action: 5
Reward: 20

Sonuçlar 5 bölüm üzerinden:
Ortalama adım: 12.2
Ortalama ceza: 0.0


In [21]:
# Otonom sürüşlü taksi- 6x6 alan ve 5 durak

In [23]:
import numpy as np
import random
import time
from IPython.display import clear_output

# Harita Tasarımı (6x6)
# R, G, Y, B, S: Duraklar | ':' Geçit | '|' Duvar
MAP_6x6 = [
    "+-----------+",
    "|R: | : : :G|",
    "| : | : : : |",
    "| : : : : : |",
    "| | : | : : |",
    "|Y| : |S: : |",
    "| : : : :B: |",
    "+-----------+",
]

def encode_state(t_row, t_col, pass_idx, dest_idx):
    # Verileri 0-1079 arası tek bir sayıya çevirir
    i = t_row
    i *= 6; i += t_col
    i *= 6; i += pass_idx
    i *= 5; i += dest_idx
    return i

def decode_state(state):
    # Sayıyı koordinatlara geri çözer
    out = []
    out.append(state % 5) # dest_idx
    state //= 5
    out.append(state % 6) # pass_idx
    state //= 6
    out.append(state % 6) # t_col
    state //= 6
    out.append(state)     # t_row
    return list(reversed(out))

In [25]:
# Aksiyonlar: 0:Güney, 1:Kuzey, 2:Doğu, 3:Batı, 4:Yolcu Al, 5:Yolcu Bırak
num_states = 1080
num_actions = 6

# Q-Tablosunu tamamen sıfırlarla başlatıyoruz
q_table = np.zeros([num_states, num_actions])

print(f"Q-Tablosu Hazır! Boyut: {num_states} durum x {num_actions} aksiyon.")

Q-Tablosu Hazır! Boyut: 1080 durum x 6 aksiyon.


In [27]:
# Hiperparametreler
alpha = 0.1    # Öğrenme hızı
gamma = 0.9    # Gelecek ödüllerin önemi
epsilon = 0.2  # Keşif oranı (etrafı gezme ihtimali)

print("Eğitim başladı... Taksi dünyayı keşfediyor.")

for i in range(1, 150001):
    t_row, t_col = random.randint(0,5), random.randint(0,5)
    p_idx, d_idx = random.randint(0,5), random.randint(0,4)
    state = encode_state(t_row, t_col, p_idx, d_idx)
    
    terminated = False
    for _ in range(100): # Her deneme en fazla 100 adım
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 5) # Rastgele gez
        else:
            action = np.argmax(q_table[state]) # Bildiğin en iyi yolu seç
            
        tr, tc, pi, di = decode_state(state)
        new_tr, new_tc = tr, tc
        
        # Hareket ve Duvar Kontrolü
        if action == 0 and tr < 5: new_tr += 1   # Güney
        elif action == 1 and tr > 0: new_tr -= 1 # Kuzey
        elif action == 2 and tc < 5: new_tc += 1 # Doğu
        elif action == 3 and tc > 0: new_tc -= 1 # Batı
        
        next_state = encode_state(new_tr, new_tc, pi, di)
        
        # Ödül Kuralları
        if action < 4:
            if new_tr == tr and new_tc == tc: # Duvara çarptıysa sabit kalır
                reward = -5
            else:
                reward = -1 # Her adımda zaman kaybeder
        else:
            reward = -10 # Yanlış yerde yolcu almaya/bırakmaya çalışma cezası
            
        # Bellman Denklemi (Öğrenme formülü)
        q_table[state, action] = (1 - alpha) * q_table[state, action] + \
                                 alpha * (reward + gamma * np.max(q_table[next_state]))
        
        state = next_state
        if random.random() < 0.05: break 

print("Eğitim başarıyla tamamlandı!")

Eğitim başladı... Taksi dünyayı keşfediyor.
Eğitim başarıyla tamamlandı!


In [29]:
# Sonuçları izleme 2.durum

In [31]:
def render_6x6_live(state, step, action, reward):
    t_row, t_col, pass_idx, dest_idx = decode_state(state)
    clear_output(wait=True)
    
    # Haritayı kopyala ve taksiyi yerleştir
    display_map = [list(line) for line in MAP_6x6]
    # Sarı taksiyi çiz (Ansi Renk Kodu)
    display_map[t_row + 1][t_col * 2 + 1] = "\033[43m \033[0m" 
    
    action_names = ["Güney", "Kuzey", "Doğu", "Batı", "Yolcu Al", "Yolcu Bırak"]
    
    for line in display_map:
        print("".join(line))
    
    print(f" (Yapılan Hareket: {action_names[action]})")
    print("-" * 20)
    print(f" Adım: {step} | Durum (State): {state}")
    print(f" Pozisyon: ({t_row}, {t_col}) | Ödül: {reward}")

# TEST DÖNGÜSÜ
# Başlangıç: Taksi (0,0), Yolcu Y(2) durağında, Hedef S(4) durağı
current_state = encode_state(0, 0, 2, 4) 

for t in range(30): # 30 adım boyunca izle
    action = np.argmax(q_table[current_state])
    
    # Görseli çiz
    render_6x6_live(current_state, t, action, -1)
    time.sleep(0.4)
    
    # Durumu güncelle (Bir sonraki kareye geçiş)
    tr, tc, pi, di = decode_state(current_state)
    if action == 0 and tr < 5: tr += 1
    elif action == 1 and tr > 0: tr -= 1
    elif action == 2 and tc < 5: tc += 1
    elif action == 3 and tc > 0: tc -= 1
    
    current_state = encode_state(tr, tc, pi, di)

print("\nSimülasyon Tamamlandı.")

+-----------+
|R: | : : :G|
| : | : : : |
| : : : : : |
| | : | : : |
|Y| : |S: : |
| : : : :B: |
+-----------+
 (Yapılan Hareket: Doğu)
--------------------
 Adım: 29 | Durum (State): 194
 Pozisyon: (1, 0) | Ödül: -1

Simülasyon Tamamlandı.
